# Section A — Local Search Algorithms

# Q1 — First-Choice & Stochastic Hill Climbing

In [11]:
import random

In [12]:
# Q1 landscape: states 1 to 12, f(s) values
# index 0 = state 1, index 1 = state 2, etc.
LANDSCAPE = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]
#             s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12

print("Q1 Landscape (states 1-12):")
print(LANDSCAPE)


Q1 Landscape (states 1-12):
[4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]


In [13]:
# helper: get left and right neighbours of a state (1-indexed)
def get_neighbours_q1(state, landscape):
    # convert to 0-based index first
    idx = state - 1
    neighbours = []

    # check left neighbour
    if idx - 1 >= 0:
        neighbours.append(state - 1)

    # check right neighbour
    if idx + 1 < len(landscape):
        neighbours.append(state + 1)

    return neighbours


In [14]:
# First-Choice Hill Climbing
# always checks left neighbour before right
# moves immediately on the first improvement found
def first_choice_hc(landscape, start):
    current = start
    path = [current]

    while True:
        # get neighbours of current state
        neighbours = get_neighbours_q1(current, landscape)
        moved = False

        # check each neighbour in order (left first, then right)
        for neighbour in neighbours:
            # is this neighbour strictly better?
            if landscape[neighbour - 1] > landscape[current - 1]:
                # yes! move there immediately
                current = neighbour
                path.append(current)
                moved = True
                break  # don't check the other neighbour

        # if we didn't move, we are stuck and stop
        if not moved:
            break

    return path, current


In [15]:
# Stochastic Hill Climbing
# collects ALL uphill neighbours first, then picks one randomly
def stochastic_hc(landscape, start):
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours_q1(current, landscape)

        # collect all strictly better neighbours
        uphill = []
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                uphill.append(neighbour)

        # if no uphill moves, we are stuck -> stop
        if len(uphill) == 0:
            break

        # pick one randomly from all uphill options
        current = random.choice(uphill)
        path.append(current)

    return path, current


In [16]:
# PART (a): run both algorithms from every starting state (1 to 12)
print("PART (a) - Running both algorithms from all 12 starting states")
print()
print("Landscape:", LANDSCAPE)
print()

# print the table header
print(f"{'Start':<7} {'Algorithm':<22} {'Path':<35} {'Terminal':<10} {'Steps'}")
print("-" * 85)

for start in range(1, 13):

    # run First-Choice HC
    path_fc, term_fc = first_choice_hc(LANDSCAPE, start)
    steps_fc = len(path_fc) - 1
    print(f"{start:<7} {'First-Choice HC':<22} {str(path_fc):<35} {term_fc:<10} {steps_fc}")

    # run Stochastic HC (fix seed so results are reproducible for part a)
    random.seed(42 + start)
    path_st, term_st = stochastic_hc(LANDSCAPE, start)
    steps_st = len(path_st) - 1
    print(f"{start:<7} {'Stochastic HC':<22} {str(path_st):<35} {term_st:<10} {steps_st}")

    print()


PART (a) - Running both algorithms from all 12 starting states

Landscape: [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]

Start   Algorithm              Path                                Terminal   Steps
-------------------------------------------------------------------------------------
1       First-Choice HC        [1, 2]                              2          1
1       Stochastic HC          [1, 2]                              2          1

2       First-Choice HC        [2]                                 2          0
2       Stochastic HC          [2]                                 2          0

3       First-Choice HC        [3, 2]                              2          1
3       Stochastic HC          [3, 4]                              4          1

4       First-Choice HC        [4]                                 4          0
4       Stochastic HC          [4]                                 4          0

5       First-Choice HC        [5, 4]                              

In [17]:
# PART (b): Analysis - how many starting states reach global max (state 11)?
print("PART (b) - Analysis + Stochastic HC 50 times from s=4")
print()

# global max is state 11 (f=16) in the Q1 landscape
GLOBAL_MAX_STATE = 11

print("[b.1] How many starting states reach global maximum (state 11, f=16)?")
print()

# count how many times each algorithm reaches state 11
fc_reach_11 = 0
st_reach_11 = 0

print(f"{'Start':<7} {'FC Terminal':<15} {'FC->11?':<10} {'ST Terminal':<15} {'ST->11?'}")
print("-" * 60)

for start in range(1, 13):

    # run First-Choice
    path_fc, term_fc = first_choice_hc(LANDSCAPE, start)

    # run Stochastic (same seed as part a for consistency)
    random.seed(42 + start)
    path_st, term_st = stochastic_hc(LANDSCAPE, start)

    # check if each reached global max
    if term_fc == GLOBAL_MAX_STATE:
        fc_reach_11 = fc_reach_11 + 1
        fc_yes = "YES"
    else:
        fc_yes = "no"

    if term_st == GLOBAL_MAX_STATE:
        st_reach_11 = st_reach_11 + 1
        st_yes = "YES"
    else:
        st_yes = "no"

    print(f"{start:<7} {term_fc:<15} {fc_yes:<10} {term_st:<15} {st_yes}")

print()
print(f"Summary: First-Choice reaches state 11 from {fc_reach_11}/12 starting states")
print(f"         Stochastic HC reaches state 11 from {st_reach_11}/12 starting states")


PART (b) - Analysis + Stochastic HC 50 times from s=4

[b.1] How many starting states reach global maximum (state 11, f=16)?

Start   FC Terminal     FC->11?    ST Terminal     ST->11?
------------------------------------------------------------
1       2               no         2               no
2       2               no         2               no
3       2               no         4               no
4       4               no         4               no
5       4               no         6               no
6       6               no         6               no
7       6               no         6               no
8       6               no         9               no
9       9               no         9               no
10      9               no         11              YES
11      11              YES        11              YES
12      11              YES        11              YES

Summary: First-Choice reaches state 11 from 2/12 starting states
         Stochastic HC reaches state 

In [18]:
# [b.2] Find starting states where FC and Stochastic reach DIFFERENT terminals
print("[b.2] Starting states where FC and Stochastic reach DIFFERENT terminals")
print()

diverge_found = False

for start in range(1, 13):

    path_fc, term_fc = first_choice_hc(LANDSCAPE, start)

    random.seed(42 + start)
    path_st, term_st = stochastic_hc(LANDSCAPE, start)

    # do they reach different terminals?
    if term_fc != term_st:
        diverge_found = True
        print(f"Start = {start}")
        print(f"  First-Choice path : {path_fc}  -> terminal state {term_fc} (f={LANDSCAPE[term_fc-1]})")
        print(f"  Stochastic path   : {path_st}  -> terminal state {term_st} (f={LANDSCAPE[term_st-1]})")

        # find the decision point: the state where both algorithms had
        # more than one uphill neighbour and made different choices
        # walk through path_fc until we find a state with 2 uphill options
        decision_state = None
        for visited_state in path_fc:
            nb = get_neighbours_q1(visited_state, LANDSCAPE)
            uphill = [n for n in nb if LANDSCAPE[n-1] > LANDSCAPE[visited_state-1]]
            if len(uphill) >= 2:
                decision_state = visited_state
                left_nb  = uphill[0]   # left neighbour always comes first
                right_nb = uphill[1]   # right neighbour comes second
                break

        if decision_state is not None:
            print(f"  Decision point: state {decision_state} (f={LANDSCAPE[decision_state-1]})")
            print(f"    Left  neighbour = state {left_nb}  (f={LANDSCAPE[left_nb-1]})")
            print(f"    Right neighbour = state {right_nb} (f={LANDSCAPE[right_nb-1]})")
            print(f"    FC always picks LEFT first -> moved to state {left_nb} (f={LANDSCAPE[left_nb-1]}), got stuck there.")
            print(f"    Stochastic randomly chose RIGHT -> moved to state {right_nb} (f={LANDSCAPE[right_nb-1]}), reached different peak.")
        print()

if not diverge_found:
    print("No divergence found. Both algorithms reached the same terminals.")


[b.2] Starting states where FC and Stochastic reach DIFFERENT terminals

Start = 3
  First-Choice path : [3, 2]  -> terminal state 2 (f=9)
  Stochastic path   : [3, 4]  -> terminal state 4 (f=11)
  Decision point: state 3 (f=6)
    Left  neighbour = state 2  (f=9)
    Right neighbour = state 4 (f=11)
    FC always picks LEFT first -> moved to state 2 (f=9), got stuck there.
    Stochastic randomly chose RIGHT -> moved to state 4 (f=11), reached different peak.

Start = 5
  First-Choice path : [5, 4]  -> terminal state 4 (f=11)
  Stochastic path   : [5, 6]  -> terminal state 6 (f=15)
  Decision point: state 5 (f=8)
    Left  neighbour = state 4  (f=11)
    Right neighbour = state 6 (f=15)
    FC always picks LEFT first -> moved to state 4 (f=11), got stuck there.
    Stochastic randomly chose RIGHT -> moved to state 6 (f=15), reached different peak.

Start = 8
  First-Choice path : [8, 7, 6]  -> terminal state 6 (f=15)
  Stochastic path   : [8, 9]  -> terminal state 9 (f=13)
  Decision 

In [19]:
# [b.3] Run Stochastic HC 50 times from s=4 (no fixed seed = true randomness)
print("[b.3] Stochastic HC run 50 times from start = 4 (no fixed seed)")
print()

reach_11 = 0
other_terminals = {}

for i in range(50):
    # no random.seed() here -- we want true randomness each time
    path_st, term_st = stochastic_hc(LANDSCAPE, 4)

    if term_st == 11:
        reach_11 = reach_11 + 1
    else:
        # record where it ended up
        if term_st in other_terminals:
            other_terminals[term_st] = other_terminals[term_st] + 1
        else:
            other_terminals[term_st] = 1

percentage = (reach_11 / 50) * 100
print(f"  Reached state 11 (global max): {reach_11}/50 times = {percentage:.1f}%")
print(f"  Other terminals reached: {other_terminals}")


[b.3] Stochastic HC run 50 times from start = 4 (no fixed seed)

  Reached state 11 (global max): 0/50 times = 0.0%
  Other terminals reached: {4: 50}


In [20]:
print("""
Explanation:
  - State 11 has f=16, which is the global maximum in the landscape.
  - First-Choice always takes the left neighbour if it is better, which can
    cause it to get trapped at local maxima (e.g., state 6 with f=15).
  - Stochastic HC picks randomly among uphill moves, so from the same start
    it may reach a different local maximum than First-Choice.
""")



Explanation:
  - State 11 has f=16, which is the global maximum in the landscape.
  - First-Choice always takes the left neighbour if it is better, which can
    cause it to get trapped at local maxima (e.g., state 6 with f=15).
  - Stochastic HC picks randomly among uphill moves, so from the same start
    it may reach a different local maximum than First-Choice.



In [21]:
print("""
Explanation:
  Divergence happens because First-Choice has a fixed bias (always checks
  left first), while Stochastic HC has randomness. From certain starts,
  First-Choice gets locked into a direction toward a local max, while
  Stochastic may randomly choose the path toward the global max.
""")



Explanation:
  Divergence happens because First-Choice has a fixed bias (always checks
  left first), while Stochastic HC has randomness. From certain starts,
  First-Choice gets locked into a direction toward a local max, while
  Stochastic may randomly choose the path toward the global max.



In [22]:
print("""
Explanation:
  From state 4 (f=11), both left neighbour s3 (f=6) and right neighbour
  s5 (f=8) are worse, so the algorithm terminates immediately at state 4
  in most runs. This reveals that Stochastic HC is unreliable when the
  starting state is already a local maximum - it cannot escape, regardless
  of how many times it is run. The randomness only helps when there are
  multiple uphill choices available.
""")



Explanation:
  From state 4 (f=11), both left neighbour s3 (f=6) and right neighbour
  s5 (f=8) are worse, so the algorithm terminates immediately at state 4
  in most runs. This reveals that Stochastic HC is unreliable when the
  starting state is already a local maximum - it cannot escape, regardless
  of how many times it is run. The randomness only helps when there are
  multiple uphill choices available.



In [23]:
# PART (c): Plateau handling
# modified landscape: states 5, 6, 7 all get f = 15
LANDSCAPE_PLATEAU = [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]
#                    s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12

print("PART (c) - Plateau Landscape: states 5, 6, 7 all have f=15")
print()
print("Modified Landscape:", LANDSCAPE_PLATEAU)
print("States 5, 6, 7 now all have f=15 (plateau region)")


PART (c) - Plateau Landscape: states 5, 6, 7 all have f=15

Modified Landscape: [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]
States 5, 6, 7 now all have f=15 (plateau region)


In [24]:
# First-Choice HC with plateau detection
# prints a WARNING when stuck on a plateau (equal neighbour but no better one)
def first_choice_hc_plateau(landscape, start):
    current = start
    path = [current]
    plateau_hit = False

    while True:
        neighbours = get_neighbours_q1(current, landscape)
        moved = False

        # first try to find a strictly better neighbour
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        # if we didn't find a better one, check for equal ones (plateau)
        if not moved:
            equal_neighbours = []
            for neighbour in neighbours:
                if landscape[neighbour - 1] == landscape[current - 1]:
                    equal_neighbours.append(neighbour)

            # if there is at least one equal neighbour -> plateau detected
            if len(equal_neighbours) > 0:
                print(f"  [WARNING] Plateau detected at state {current} (f={landscape[current-1]}). No strictly better neighbour.")
                plateau_hit = True

            break  # stop either way (no better neighbour found)

    return path, current, plateau_hit


In [25]:
# Stochastic HC with plateau detection
def stochastic_hc_plateau(landscape, start):
    current = start
    path = [current]
    plateau_hit = False

    while True:
        neighbours = get_neighbours_q1(current, landscape)

        # collect strictly better neighbours
        uphill = []
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                uphill.append(neighbour)

        if len(uphill) > 0:
            # move to a random uphill neighbour
            current = random.choice(uphill)
            path.append(current)
        else:
            # no uphill move, check if there is an equal neighbour (plateau)
            equal_neighbours = []
            for neighbour in neighbours:
                if landscape[neighbour - 1] == landscape[current - 1]:
                    equal_neighbours.append(neighbour)

            if len(equal_neighbours) > 0:
                print(f"  [WARNING] Plateau detected at state {current} (f={landscape[current-1]}). No strictly better neighbour.")
                plateau_hit = True

            break  # stop

    return path, current, plateau_hit


In [26]:
# First-Choice HC with sideways moves (max 10 sideways moves per run)
def first_choice_hc_sideways(landscape, start, max_sideways=10):
    current = start
    path = [current]
    sideways_count = 0  # keep track of how many sideways moves we made

    while True:
        neighbours = get_neighbours_q1(current, landscape)
        moved = False

        # first try to find a strictly better neighbour (uphill move)
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        # if no uphill move, try sideways (equal) if we have budget left
        if not moved and sideways_count < max_sideways:
            equal_neighbours = []
            for neighbour in neighbours:
                if landscape[neighbour - 1] == landscape[current - 1]:
                    equal_neighbours.append(neighbour)

            if len(equal_neighbours) > 0:
                # pick the highest-indexed equal neighbour (move rightward to cross plateau)
                best_equal = equal_neighbours[0]
                for n in equal_neighbours:
                    if n > best_equal:
                        best_equal = n
                current = best_equal
                path.append(current)
                sideways_count = sideways_count + 1
                moved = True

        # if still no move, we are done
        if not moved:
            break

    return path, current


In [27]:
# Stochastic HC with sideways moves (max 10 sideways moves per run)
def stochastic_hc_sideways(landscape, start, max_sideways=10):
    current = start
    path = [current]
    sideways_count = 0

    while True:
        neighbours = get_neighbours_q1(current, landscape)

        # collect strictly better neighbours
        uphill = []
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                uphill.append(neighbour)

        if len(uphill) > 0:
            # pick a random uphill neighbour
            current = random.choice(uphill)
            path.append(current)

        elif sideways_count < max_sideways:
            # no uphill, try sideways
            equal_neighbours = []
            for neighbour in neighbours:
                if landscape[neighbour - 1] == landscape[current - 1]:
                    equal_neighbours.append(neighbour)

            if len(equal_neighbours) > 0:
                current = random.choice(equal_neighbours)
                path.append(current)
                sideways_count = sideways_count + 1
            else:
                break  # no equal neighbours either -> truly stuck
        else:
            break  # used up all sideways moves

    return path, current


In [29]:
# Part (c): Run WITHOUT sideways moves - plateau detection
print("[c.1] WITHOUT sideways moves - Plateau Detection")
print()

fc_plateau_stuck = 0
st_plateau_stuck = 0

print("-- First-Choice HC with plateau detection --")
for start in range(1, 13):
    path, term, hit = first_choice_hc_plateau(LANDSCAPE_PLATEAU, start)
    if hit:
        fc_plateau_stuck = fc_plateau_stuck + 1
    if hit:
        plateau_str = "YES"
    else:
        plateau_str = "no"
    print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), plateau={plateau_str}")

print(f"\n  => First-Choice HC got stuck on plateau in {fc_plateau_stuck}/12 runs")

[c.1] WITHOUT sideways moves - Plateau Detection

-- First-Choice HC with plateau detection --
  Start=1: path=[1, 2], terminal=2 (f=9), plateau=no
  Start=2: path=[2], terminal=2 (f=9), plateau=no
  Start=3: path=[3, 2], terminal=2 (f=9), plateau=no
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=4: path=[4, 5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=5: path=[5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 6 (f=15). No strictly better neighbour.
  Start=6: path=[6], terminal=6 (f=15), plateau=YES
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  Start=7: path=[7], terminal=7 (f=15), plateau=YES
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  Start=8: path=[8, 7], terminal=7 (f=15), plateau=YES
  Start=9: path=[9], terminal=9 (f=13), plateau=no
  Start=10: path=[10, 9], terminal=9 (f=13

In [30]:
print("-- Stochastic HC with plateau detection --")
for start in range(1, 13):
    random.seed(42 + start)
    path, term, hit = stochastic_hc_plateau(LANDSCAPE_PLATEAU, start)
    if hit:
        st_plateau_stuck = st_plateau_stuck + 1
    if hit:
        plateau_str = "YES"
    else:
        plateau_str = "no"
    print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), plateau={plateau_str}")

print(f"\n  => Stochastic HC got stuck on plateau in {st_plateau_stuck}/12 runs")


-- Stochastic HC with plateau detection --
  Start=1: path=[1, 2], terminal=2 (f=9), plateau=no
  Start=2: path=[2], terminal=2 (f=9), plateau=no
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=3: path=[3, 4, 5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=4: path=[4, 5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  Start=5: path=[5], terminal=5 (f=15), plateau=YES
  [WARNING] Plateau detected at state 6 (f=15). No strictly better neighbour.
  Start=6: path=[6], terminal=6 (f=15), plateau=YES
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  Start=7: path=[7], terminal=7 (f=15), plateau=YES
  Start=8: path=[8, 9], terminal=9 (f=13), plateau=no
  Start=9: path=[9], terminal=9 (f=13), plateau=no
  Start=10: path=[10, 11], terminal=11 (f=16), plateau=no
  Start=11: path=[11], terminal=

In [31]:
# Part (c): Run WITH sideways moves (cap = 10)
print("[c.2] WITH sideways moves (cap = 10)")
print()

fc_success = 0  # how many times reached global max (state 11)
st_success = 0

print("-- First-Choice HC with sideways moves --")
for start in range(1, 13):
    path, term = first_choice_hc_sideways(LANDSCAPE_PLATEAU, start)
    if term == 11:
        fc_success = fc_success + 1
        reached_str = "YES"
    else:
        reached_str = "no"
    print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), reached_11={reached_str}")

print(f"\n  => First-Choice HC with sideways: reached global max in {fc_success}/12 runs")

[c.2] WITH sideways moves (cap = 10)

-- First-Choice HC with sideways moves --
  Start=1: path=[1, 2], terminal=2 (f=9), reached_11=no
  Start=2: path=[2], terminal=2 (f=9), reached_11=no
  Start=3: path=[3, 2], terminal=2 (f=9), reached_11=no
  Start=4: path=[4, 5, 6, 7, 6, 7, 6, 7, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=5: path=[5, 6, 7, 6, 7, 6, 7, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=6: path=[6, 7, 6, 7, 6, 7, 6, 7, 6, 7, 6], terminal=6 (f=15), reached_11=no
  Start=7: path=[7, 6, 7, 6, 7, 6, 7, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=8: path=[8, 7, 6, 7, 6, 7, 6, 7, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=9: path=[9], terminal=9 (f=13), reached_11=no
  Start=10: path=[10, 9], terminal=9 (f=13), reached_11=no
  Start=11: path=[11], terminal=11 (f=16), reached_11=YES
  Start=12: path=[12, 11], terminal=11 (f=16), reached_11=YES

  => First-Choice HC with sideways: reached global max in 2/12 runs


In [32]:
print("-- Stochastic HC with sideways moves --")
for start in range(1, 13):
    random.seed(42 + start)
    path, term = stochastic_hc_sideways(LANDSCAPE_PLATEAU, start)
    if term == 11:
        st_success = st_success + 1
        reached_str = "YES"
    else:
        reached_str = "no"
    print(f"  Start={start}: path={path}, terminal={term} (f={LANDSCAPE_PLATEAU[term-1]}), reached_11={reached_str}")

print(f"\n  => Stochastic HC with sideways: reached global max in {st_success}/12 runs")

-- Stochastic HC with sideways moves --
  Start=1: path=[1, 2], terminal=2 (f=9), reached_11=no
  Start=2: path=[2], terminal=2 (f=9), reached_11=no
  Start=3: path=[3, 4, 5, 6, 7, 6, 7, 6, 5, 6, 7, 6, 5], terminal=5 (f=15), reached_11=no
  Start=4: path=[4, 5, 6, 5, 6, 5, 6, 5, 6, 7, 6, 5], terminal=5 (f=15), reached_11=no
  Start=5: path=[5, 6, 5, 6, 7, 6, 7, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=6: path=[6, 7, 6, 7, 6, 7, 6, 5, 6, 7, 6], terminal=6 (f=15), reached_11=no
  Start=7: path=[7, 6, 7, 6, 5, 6, 5, 6, 7, 6, 7], terminal=7 (f=15), reached_11=no
  Start=8: path=[8, 9], terminal=9 (f=13), reached_11=no
  Start=9: path=[9], terminal=9 (f=13), reached_11=no
  Start=10: path=[10, 11], terminal=11 (f=16), reached_11=YES
  Start=11: path=[11], terminal=11 (f=16), reached_11=YES
  Start=12: path=[12, 11], terminal=11 (f=16), reached_11=YES

  => Stochastic HC with sideways: reached global max in 3/12 runs


In [33]:
# re-count plateau stucks so summary is always correct even if cells run independently
fc_plateau_stuck_check = 0
st_plateau_stuck_check = 0
for start_check in range(1, 13):
    _, _, hit_fc = first_choice_hc_plateau(LANDSCAPE_PLATEAU, start_check)
    if hit_fc:
        fc_plateau_stuck_check = fc_plateau_stuck_check + 1
    random.seed(42 + start_check)
    _, _, hit_st = stochastic_hc_plateau(LANDSCAPE_PLATEAU, start_check)
    if hit_st:
        st_plateau_stuck_check = st_plateau_stuck_check + 1

print()
print("Summary comparison:")
print(f"  Without sideways: FC stuck in {fc_plateau_stuck_check}/12, Stochastic stuck in {st_plateau_stuck_check}/12")
print(f"  With sideways:    FC reached global max in {fc_success}/12, Stochastic in {st_success}/12")


  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 5 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 6 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 6 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.
  [WARNING] Plateau detected at state 7 (f=15). No strictly better neighbour.

Summary comparison:
  Without sideways: FC stuck in 5/12, Stochastic stuck in 5/12
  With sideways:    FC reached global max in 2/12, Stochastic in 3/12


In [34]:
print("""
Explanation (without sideways moves):
  When states 5, 6, 7 all have f=15, any algorithm reaching one of them
  sees equal-valued neighbours but no strictly better neighbour. Without
  sideways moves, the algorithm simply stops at the edge of the plateau,
  unable to cross it and reach the true global maximum at state 11 (f=16).
""")



Explanation (without sideways moves):
  When states 5, 6, 7 all have f=15, any algorithm reaching one of them
  sees equal-valued neighbours but no strictly better neighbour. Without
  sideways moves, the algorithm simply stops at the edge of the plateau,
  unable to cross it and reach the true global maximum at state 11 (f=16).



In [35]:
print("""
Explanation (with sideways moves - 4 to 5 sentences):
  The sideways-move extension allows both algorithms to traverse the plateau
  (states 5, 6, 7 with f=15) instead of stopping at its edge. First-Choice HC
  benefits significantly because it always picks the leftmost equal neighbour,
  which gives it a consistent direction to cross the plateau toward state 8
  and eventually reach state 11. Stochastic HC also benefits, but it picks a
  random equal neighbour, meaning it may wander on the plateau and waste
  sideways-move budget without making progress. The cap of 10 is important
  because without it, the algorithm could loop forever on a flat plateau
  (especially Stochastic HC, which picks randomly).
""")



Explanation (with sideways moves - 4 to 5 sentences):
  The sideways-move extension allows both algorithms to traverse the plateau
  (states 5, 6, 7 with f=15) instead of stopping at its edge. First-Choice HC
  benefits significantly because it always picks the leftmost equal neighbour,
  which gives it a consistent direction to cross the plateau toward state 8
  and eventually reach state 11. Stochastic HC also benefits, but it picks a
  random equal neighbour, meaning it may wander on the plateau and waste
  sideways-move budget without making progress. The cap of 10 is important
  because without it, the algorithm could loop forever on a flat plateau
  (especially Stochastic HC, which picks randomly).



# Q2 — Random Restart Hill Climbing

In [1]:
import random

In [2]:
# Q2 landscape: 14 states (warehouse robot scenario)
LANDSCAPE_Q2 = [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
#               s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12 s13 s14

# global maximum is state 11, f(11) = 19
print("Q2 Landscape (14 states):", LANDSCAPE_Q2)
print("Global maximum: state 11, f(11) = 19")


Q2 Landscape (14 states): [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Global maximum: state 11, f(11) = 19


In [3]:
# helper: get left and right neighbours (1-indexed, for Q2)
def get_neighbours_q2(state, landscape):
    idx = state - 1
    neighbours = []

    # check left neighbour
    if idx - 1 >= 0:
        neighbours.append(state - 1)

    # check right neighbour
    if idx + 1 < len(landscape):
        neighbours.append(state + 1)

    return neighbours


In [4]:
# First-Choice HC for Q2 (same logic as Q1)
def first_choice_hc_q2(landscape, start):
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours_q2(current, landscape)
        moved = False

        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        if not moved:
            break

    return path, current

In [5]:
# Stochastic HC for Q2
def stochastic_hc_q2(landscape, start):
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours_q2(current, landscape)

        # collect all uphill neighbours
        uphill = []
        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                uphill.append(neighbour)

        if len(uphill) == 0:
            break

        current = random.choice(uphill)
        path.append(current)

    return path, current


In [6]:
# find_local_maxima: returns all states whose f-value beats both neighbours
def find_local_maxima_q2(landscape):
    local_maxima = []
    n = len(landscape)

    for i in range(n):
        state = i + 1  # 1-indexed state
        val = landscape[i]

        # check left side: boundary states only need to beat one side
        if i == 0:
            left_ok = True   # no left neighbour at boundary
        else:
            left_ok = (val > landscape[i - 1])

        # check right side
        if i == n - 1:
            right_ok = True  # no right neighbour at boundary
        else:
            right_ok = (val > landscape[i + 1])

        # only add if it beats BOTH sides
        if left_ok and right_ok:
            local_maxima.append(state)

    return local_maxima


In [7]:
# Random Restart Hill Climbing
def random_restart_hc_q2(landscape, num_restarts, variant="first_choice"):
    best_state = None
    best_value = -1
    all_results = []  # list of (start, terminal, path)

    for i in range(num_restarts):
        # pick a random starting state (1-indexed)
        start = random.randint(0, len(landscape) - 1) + 1

        # run HC with chosen variant
        if variant == "first_choice":
            path, terminal = first_choice_hc_q2(landscape, start)
        else:
            path, terminal = stochastic_hc_q2(landscape, start)

        terminal_value = landscape[terminal - 1]
        all_results.append((start, terminal, path))

        # update best if this restart found something better
        if terminal_value > best_value:
            best_value = terminal_value
            best_state = terminal

    return best_state, best_value, all_results


In [8]:
# PART (a): find local maxima + run RRHC with 20 restarts
print("PART (a) - Random Restart HC Implementation")
print()
print(f"Landscape (14 states): {LANDSCAPE_Q2}")
print("Global maximum: state 11, f(11) = 19")
print()

# find and print all local maxima
local_maxima = find_local_maxima_q2(LANDSCAPE_Q2)
print("[a.1] Local Maxima in Q2 Landscape:")
print("-" * 45)
for s in local_maxima:
    print(f"  State {s:>2} -> f({s}) = {LANDSCAPE_Q2[s-1]}")
print(f"\n  Total local maxima found: {len(local_maxima)}")

print()

PART (a) - Random Restart HC Implementation

Landscape (14 states): [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Global maximum: state 11, f(11) = 19

[a.1] Local Maxima in Q2 Landscape:
---------------------------------------------
  State  2 -> f(2) = 8
  State  4 -> f(4) = 12
  State  7 -> f(7) = 17
  State 11 -> f(11) = 19

  Total local maxima found: 4



In [9]:
# run RRHC with 20 restarts - First-Choice
print("[a.2] RRHC with num_restarts=20 -- First-Choice variant")
random.seed(100)
best_s, best_v, results_fc = random_restart_hc_q2(LANDSCAPE_Q2, 20, variant="first_choice")

global_max_state = 11

print(f"{'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'Global Max Found?'}")
print("-" * 55)
for i in range(len(results_fc)):
    start = results_fc[i][0]
    terminal = results_fc[i][1]
    path = results_fc[i][2]
    if terminal == global_max_state:
        found = "YES"
    else:
        found = "no"
    print(f"{i+1:<10} {start:<8} {terminal:<12} {LANDSCAPE_Q2[terminal-1]:<10} {found}")

print(f"\n  Best state found: {best_s}  |  Best f-value: {best_v}")
if best_s == global_max_state:
    print("  Global max (state 11) found: YES")
else:
    print("  Global max (state 11) found: NO")

[a.2] RRHC with num_restarts=20 -- First-Choice variant
Restart    Start    Terminal     f-value    Global Max Found?
-------------------------------------------------------
1          3        2            8          no
2          8        7            17         no
3          8        7            17         no
4          13       11           19         YES
5          3        2            8          no
6          12       11           19         YES
7          7        7            17         no
8          12       11           19         YES
9          6        4            12         no
10         7        7            17         no
11         9        7            17         no
12         13       11           19         YES
13         2        2            8          no
14         9        7            17         no
15         2        2            8          no
16         2        2            8          no
17         12       11           19         YES
18         8        7 

In [10]:
# run RRHC with 20 restarts - Stochastic
print("[a.3] RRHC with num_restarts=20 -- Stochastic variant")
random.seed(200)
best_s2, best_v2, results_st = random_restart_hc_q2(LANDSCAPE_Q2, 20, variant="stochastic")

print(f"{'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'Global Max Found?'}")
for i in range(len(results_st)):
    start = results_st[i][0]
    terminal = results_st[i][1]
    if terminal == global_max_state:
        found = "YES"
    else:
        found = "no"
    print(f"{i+1:<10} {start:<8} {terminal:<12} {LANDSCAPE_Q2[terminal-1]:<10} {found}")

print(f"\n  Best state found: {best_s2}  |  Best f-value: {best_v2}")
if best_s2 == global_max_state:
    print("  Global max (state 11) found: YES")
else:
    print("  Global max (state 11) found: NO")


[a.3] RRHC with num_restarts=20 -- Stochastic variant
Restart    Start    Terminal     f-value    Global Max Found?
1          1        2            8          no
2          12       11           19         YES
3          3        4            12         no
4          1        2            8          no
5          12       11           19         YES
6          12       11           19         YES
7          8        7            17         no
8          8        7            17         no
9          4        4            12         no
10         4        4            12         no
11         11       11           19         YES
12         14       11           19         YES
13         6        4            12         no
14         7        7            17         no
15         4        4            12         no
16         11       11           19         YES
17         8        7            17         no
18         11       11           19         YES
19         12       11         

In [ ]:
print("""
Explanation:
  A local maximum is any state whose f-value is strictly greater than
  both its neighbours. In this 14-state landscape we have multiple local
  peaks, which is exactly why basic HC gets trapped -- it climbs to the
  nearest peak and stops, never seeing the true global max at state 11.
""")



Explanation:
  A local maximum is any state whose f-value is strictly greater than
  both its neighbours. In this 14-state landscape we have multiple local
  peaks, which is exactly why basic HC gets trapped -- it climbs to the
  nearest peak and stops, never seeing the true global max at state 11.



In [ ]:
print("""
Explanation:
  Random Restart solves the local maximum problem by simply re-running HC
  from a fresh random start whenever it gets stuck. Over many restarts,
  the algorithm is likely to eventually start near the global maximum and
  climb to it. First-Choice is deterministic so the same start always gives
  the same terminal; Stochastic adds randomness within each run as well.
""")



Explanation:
  Random Restart solves the local maximum problem by simply re-running HC
  from a fresh random start whenever it gets stuck. Over many restarts,
  the algorithm is likely to eventually start near the global maximum and
  climb to it. First-Choice is deterministic so the same start always gives
  the same terminal; Stochastic adds randomness within each run as well.



In [ ]:
# PART (b): Empirical vs Theoretical Probability
print("PART (b) - Empirical vs Theoretical Probability of Finding Global Max")
print()

global_max_state = 11
restart_counts = [1, 3, 5, 10, 20]

# Step 1: derive p = fraction of starts that reach state 11 via First-Choice HC
print("[b.1] Deriving p: fraction of starts reaching state 11 via First-Choice HC")
print("-" * 60)
print()

reach_count = 0
total_states = len(LANDSCAPE_Q2)

print(f"{'Start':<8} {'Terminal':<12} {'Reached 11?'}")
print("-" * 35)

for start in range(1, total_states + 1):
    path, terminal = first_choice_hc_q2(LANDSCAPE_Q2, start)
    if terminal == global_max_state:
        reach_count = reach_count + 1
        reached_str = "YES"
    else:
        reached_str = "no"
    print(f"{start:<8} {terminal:<12} {reached_str}")

# p is the probability that a random start reaches state 11
p = reach_count / total_states
print(f"\n  States reaching global max: {reach_count} / {total_states}")
print(f"  p = {reach_count}/{total_states} = {p:.4f}")


PART (b) - Empirical vs Theoretical Probability of Finding Global Max

[b.1] Deriving p: fraction of starts reaching state 11 via First-Choice HC
------------------------------------------------------------

Start    Terminal     Reached 11?
-----------------------------------
1        2            no
2        2            no
3        2            no
4        4            no
5        4            no
6        4            no
7        7            no
8        7            no
9        7            no
10       7            no
11       11           YES
12       11           YES
13       11           YES
14       11           YES

  States reaching global max: 4 / 14
  p = 4/14 = 0.2857


In [ ]:
# Step 2: Empirical probabilities (100 independent trials each)
print()
print("[b.2] Empirical Probability: 100 independent trials per restart count n")
print()

empirical_probs = {}
TRIALS = 100

# go through each value of n
for n in restart_counts:
    success_count = 0

    for trial in range(TRIALS):
        # give each trial its own seed so they are independent
        random.seed(trial * 1000 + n)
        best_s, best_v, _ = random_restart_hc_q2(LANDSCAPE_Q2, n, variant="first_choice")
        if best_s == global_max_state:
            success_count = success_count + 1

    prob = success_count / TRIALS
    empirical_probs[n] = prob

print(f"{'n (restarts)':<15} {'Successes/100':<18} {'Empirical P'}")
print("-" * 45)
for n in restart_counts:
    prob = empirical_probs[n]
    successes = int(prob * TRIALS)
    print(f"{n:<15} {successes:<18} {prob:.4f}")



[b.2] Empirical Probability: 100 independent trials per restart count n

n (restarts)    Successes/100      Empirical P
---------------------------------------------
1               31                 0.3100
3               56                 0.5600
5               83                 0.8300
10              97                 0.9700
20              99                 0.9900


In [ ]:
# Step 3: Theoretical probability P = 1 - (1 - p)^n
print()
print("[b.3] Theoretical Probability: P = 1 - (1 - p)^n")
print(f"  Using p = {p:.4f} (derived above)")
print()

theoretical_probs = {}

print(f"  {'n':<6} {'Calculation':<35} {'Theoretical P'}")
print("  " + "-" * 55)

for n in restart_counts:
    # P = 1 - (1 - p)^n
    failure_prob = (1 - p) ** n
    theo = 1 - failure_prob
    theoretical_probs[n] = theo
    calc_str = f"1 - (1 - {p:.4f})^{n}"
    print(f"  {n:<6} {calc_str:<35} {theo:.4f}")

# Step 4: comparison table
print()
print("[b.4] Comparison: Empirical vs Theoretical")
print(f"{'n':<8} {'Empirical P':<15} {'Theoretical P':<15} {'Difference'}")
print("-" * 50)

for n in restart_counts:
    emp = empirical_probs[n]
    theo = theoretical_probs[n]
    diff = abs(emp - theo)
    print(f"{n:<8} {emp:<15.4f} {theo:<15.4f} {diff:.4f}")



[b.3] Theoretical Probability: P = 1 - (1 - p)^n
  Using p = 0.2857 (derived above)

  n      Calculation                         Theoretical P
  -------------------------------------------------------
  1      1 - (1 - 0.2857)^1                  0.2857
  3      1 - (1 - 0.2857)^3                  0.6356
  5      1 - (1 - 0.2857)^5                  0.8141
  10     1 - (1 - 0.2857)^10                 0.9654
  20     1 - (1 - 0.2857)^20                 0.9988

[b.4] Comparison: Empirical vs Theoretical
n        Empirical P     Theoretical P   Difference
--------------------------------------------------
1        0.3100          0.2857          0.0243
3        0.5600          0.6356          0.0756
5        0.8300          0.8141          0.0159
10       0.9700          0.9654          0.0046
20       0.9900          0.9988          0.0088


In [ ]:
print("""
Explanation:
  p is the probability that a single random restart finds the global max.
  We compute it by running First-Choice HC from every possible start and
  counting how many reach state 11. This p is used in the theoretical formula.
""")



Explanation:
  p is the probability that a single random restart finds the global max.
  We compute it by running First-Choice HC from every possible start and
  counting how many reach state 11. This p is used in the theoretical formula.



In [ ]:
print("""
Explanation:
  For each n, we run RRHC 100 times independently and count how often
  state 11 is found. Dividing by 100 gives the empirical probability.
  As n grows, more restarts = more chances to land near state 11.
""")



Explanation:
  For each n, we run RRHC 100 times independently and count how often
  state 11 is found. Dividing by 100 gives the empirical probability.
  As n grows, more restarts = more chances to land near state 11.



In [ ]:
print("""
Explanation:
  The formula P = 1-(1-p)^n models the probability of finding the global
  max in at least one of n independent restarts, assuming each restart has
  probability p of success. This is a geometric complement: (1-p)^n is the
  chance ALL restarts fail, so 1 minus that is the chance at least one works.
""")



Explanation:
  The formula P = 1-(1-p)^n models the probability of finding the global
  max in at least one of n independent restarts, assuming each restart has
  probability p of success. This is a geometric complement: (1-p)^n is the
  chance ALL restarts fail, so 1 minus that is the chance at least one works.



In [ ]:
print("""
Explanation (3-4 sentences):
  The empirical and theoretical probabilities are generally close, especially
  for small n, which validates the independence assumption behind the formula.
  As n increases the theoretical formula predicts higher probabilities more
  optimistically because it assumes each restart is fully independent and p
  is perfectly constant -- but in practice, random starts are not uniformly
  distributed across all 14 states in every trial, causing small deviations.
  Any gap at larger n reveals that the formula's assumption of a fixed p
  breaks down slightly when the landscape has clusters of local maxima that
  bias random starts away from the global max basin of attraction.
""")



Explanation (3-4 sentences):
  The empirical and theoretical probabilities are generally close, especially
  for small n, which validates the independence assumption behind the formula.
  As n increases the theoretical formula predicts higher probabilities more
  optimistically because it assumes each restart is fully independent and p
  is perfectly constant -- but in practice, random starts are not uniformly
  distributed across all 14 states in every trial, causing small deviations.
  Any gap at larger n reveals that the formula's assumption of a fixed p
  breaks down slightly when the landscape has clusters of local maxima that
  bias random starts away from the global max basin of attraction.



In [ ]:
# PART (c): Plateau Investigation
# modified landscape: states 7 and 8 both get f = 17
LANDSCAPE_Q2_PLATEAU = [5, 8, 6, 12, 9, 7, 17, 17, 10, 6, 19, 15, 11, 8]
#                       s1  s2  s3  s4  s5  s6  s7  s8  s9 s10 s11 s12 s13 s14

print("PART (c) - Plateau Investigation: states 7 and 8 both have f=17")
print()
print("Original Landscape :", LANDSCAPE_Q2)
print("Modified Landscape :", LANDSCAPE_Q2_PLATEAU)
print("Change: states 7 and 8 now both have f=17 (plateau region)")


PART (c) - Plateau Investigation: states 7 and 8 both have f=17

Original Landscape : [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
Modified Landscape : [5, 8, 6, 12, 9, 7, 17, 17, 10, 6, 19, 15, 11, 8]
Change: states 7 and 8 now both have f=17 (plateau region)


In [ ]:
# RRHC with plateau counter
# this version also counts how many restarts terminate on the plateau
def random_restart_hc_plateau_q2(landscape, num_restarts, variant="first_choice", plateau_states=None):
    if plateau_states is None:
        plateau_states = []

    best_state = None
    best_value = -1
    all_results = []
    plateau_terminations = 0

    for i in range(num_restarts):
        start = random.randint(0, len(landscape) - 1) + 1

        if variant == "first_choice":
            path, terminal = first_choice_hc_q2(landscape, start)
        else:
            path, terminal = stochastic_hc_q2(landscape, start)

        terminal_value = landscape[terminal - 1]
        all_results.append((start, terminal, path))

        # check if this restart ended up on the plateau
        if terminal in plateau_states:
            plateau_terminations = plateau_terminations + 1

        if terminal_value > best_value:
            best_value = terminal_value
            best_state = terminal

    return best_state, best_value, all_results, plateau_terminations


In [ ]:
# run BEFORE plateau modification
print("[c.1] BEFORE plateau -- original landscape, 20 restarts")

global_max_state = 11
plateau_states = [7, 8]
NUM_RESTARTS = 20
TRIALS = 50

random.seed(300)
fc_found_before = 0
for i in range(TRIALS):
    best_s, _, _, _ = random_restart_hc_plateau_q2(LANDSCAPE_Q2, NUM_RESTARTS, variant="first_choice", plateau_states=[])
    if best_s == global_max_state:
        fc_found_before = fc_found_before + 1

rate_before = fc_found_before / TRIALS
print(f"  Global max discovery rate (original landscape): {fc_found_before}/{TRIALS} = {rate_before:.2%}")

# run AFTER plateau modification
print()
print("[c.2] AFTER plateau modification -- states 7 & 8 both have f=17")

# show a single detailed run of 20 restarts
print()
print("  Restart-by-restart detail (one run of 20 restarts shown):")
print(f"  {'Restart':<10} {'Start':<8} {'Terminal':<12} {'f-value':<10} {'On Plateau?':<14} {'Global Max?'}")

random.seed(400)
_, _, detail_results, _ = random_restart_hc_plateau_q2(LANDSCAPE_Q2_PLATEAU, NUM_RESTARTS, variant="first_choice", plateau_states=plateau_states)

for i in range(len(detail_results)):
    start = detail_results[i][0]
    terminal = detail_results[i][1]

    if terminal in plateau_states:
        on_plateau = "YES"
    else:
        on_plateau = "no"

    if terminal == global_max_state:
        found_global = "YES"
    else:
        found_global = "no"

    print(f"  {i+1:<10} {start:<8} {terminal:<12} {LANDSCAPE_Q2_PLATEAU[terminal-1]:<10} {on_plateau:<14} {found_global}")

# now do 50 trials to get stable rates
random.seed(300)
fc_found_after = 0
total_plateau_terminations = 0

for i in range(TRIALS):
    best_s, _, _, plateau_count = random_restart_hc_plateau_q2(LANDSCAPE_Q2_PLATEAU, NUM_RESTARTS, variant="first_choice", plateau_states=plateau_states)
    if best_s == global_max_state:
        fc_found_after = fc_found_after + 1
    total_plateau_terminations = total_plateau_terminations + plateau_count

rate_after = fc_found_after / TRIALS
avg_plateau = total_plateau_terminations / TRIALS

print(f"\n  Global max discovery rate (plateau landscape): {fc_found_after}/{TRIALS} = {rate_after:.2%}")
print(f"  Average plateau terminations per run of 20 restarts: {avg_plateau:.1f}")

print()
print("[c.3] Summary Comparison Table")
print(f"  {'Landscape':<25} {'Discovery Rate':<18} {'Avg Plateau Stuck'}")
print(f"  {'Original (no plateau)':<25} {rate_before:<18.2%} {'N/A'}")
print(f"  {'With plateau (s7=s8=17)':<25} {rate_after:<18.2%} {avg_plateau:.1f}/20 restarts")


[c.1] BEFORE plateau -- original landscape, 20 restarts
  Global max discovery rate (original landscape): 50/50 = 100.00%

[c.2] AFTER plateau modification -- states 7 & 8 both have f=17

  Restart-by-restart detail (one run of 20 restarts shown):
  Restart    Start    Terminal     f-value    On Plateau?    Global Max?
  1          5        4            12         no             no
  2          9        8            17         YES            no
  3          5        4            12         no             no
  4          13       11           19         no             YES
  5          14       11           19         no             YES
  6          2        2            8          no             no
  7          10       8            17         YES            no
  8          8        8            17         YES            no
  9          13       11           19         no             YES
  10         13       11           19         no             YES
  11         8        8            

In [ ]:
print("""
Explanation (3-4 sentences):
  After setting states 7 and 8 to f=17, many restarts that start between
  states 6 and 9 climb up to the plateau at f=17 and stop there, because
  neither neighbour is strictly better (state 6 has f=7, state 9 has f=10,
  and the other plateau state has the same f=17). This directly reduces the
  global max discovery rate compared to the original landscape, because those
  restarts "waste" their climb on the plateau rather than reaching state 11.
  Running more restarts does eventually increase the chance of finding state
  11, but the plateau acts as a persistent trap -- it absorbs restarts that
  start in a wide basin, so the improvement with more restarts is slower than
  in the original landscape without the plateau.
""")



Explanation (3-4 sentences):
  After setting states 7 and 8 to f=17, many restarts that start between
  states 6 and 9 climb up to the plateau at f=17 and stop there, because
  neither neighbour is strictly better (state 6 has f=7, state 9 has f=10,
  and the other plateau state has the same f=17). This directly reduces the
  global max discovery rate compared to the original landscape, because those
  restarts "waste" their climb on the plateau rather than reaching state 11.
  Running more restarts does eventually increase the chance of finding state
  11, but the plateau acts as a persistent trap -- it absorbs restarts that
  start in a wide basin, so the improvement with more restarts is slower than
  in the original landscape without the plateau.



# Q3 — Diagnosing & Fixing HC Failures + N-Queens

In [ ]:
import random

In [ ]:
# helper: get left and right neighbours (1-indexed, reused for Q3)
def get_neighbours_q3(state, landscape):
    idx = state - 1
    neighbours = []

    if idx - 1 >= 0:
        neighbours.append(state - 1)

    if idx + 1 < len(landscape):
        neighbours.append(state + 1)

    return neighbours

In [ ]:
# First-Choice HC for Q3 (basic version)
def first_choice_hc_q3(landscape, start):
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours_q3(current, landscape)
        moved = False

        for neighbour in neighbours:
            if landscape[neighbour - 1] > landscape[current - 1]:
                current = neighbour
                path.append(current)
                moved = True
                break

        if not moved:
            break

    return path, current


In [ ]:
# PART (a): diagnose_hc - detects which failure mode occurred
def diagnose_hc(landscape, start):
    """
    Runs First-Choice HC and automatically diagnoses failure mode:
    1. LOCAL MAXIMUM - no better AND no equal neighbour exists
    2. PLATEAU       - no strictly better but at least one equal neighbour
    3. RIDGE         - stuck at sub-optimal peak with a valley between
                       current position and the global maximum
    """
    current = start
    path = [current]

    while True:
        neighbours = get_neighbours_q3(current, landscape)

        # separate neighbours into better, equal, and worse
        better_neighbours = []
        equal_neighbours = []

        for n in neighbours:
            if landscape[n - 1] > landscape[current - 1]:
                better_neighbours.append(n)
            elif landscape[n - 1] == landscape[current - 1]:
                equal_neighbours.append(n)

        # if there is a better neighbour, move there (First-Choice)
        if len(better_neighbours) > 0:
            current = better_neighbours[0]  # pick leftmost better
            path.append(current)

        elif len(equal_neighbours) > 0:
            # no better but there is an equal neighbour -> PLATEAU
            print(f"  Terminated at state {current} with f={landscape[current-1]}. Failure mode: PLATEAU")
            return path, current, "plateau"

        else:
            # no better, no equal -> stuck completely
            # check if this is the global max or a ridge
            global_max_val = max(landscape)
            current_val = landscape[current - 1]

            if current_val < global_max_val:
                # we are NOT at the global max -> check if a valley separates us from it
                global_max_pos = 0
                for idx in range(len(landscape)):
                    if landscape[idx] == global_max_val:
                        global_max_pos = idx + 1  # 1-indexed

                lo = min(current, global_max_pos)
                hi = max(current, global_max_pos)

                # look for a valley (state lower than BOTH endpoints) between lo and hi
                valley_found = False
                for idx in range(lo, hi - 1):  # states between lo and hi
                    v = landscape[idx]
                    if v < landscape[lo - 1] and v < landscape[hi - 1]:
                        valley_found = True
                        break

                if valley_found:
                    print(f"  Terminated at state {current} with f={landscape[current-1]}. Failure mode: RIDGE")
                    return path, current, "ridge"

            # no valley or already at global max -> LOCAL MAXIMUM
            print(f"  Terminated at state {current} with f={landscape[current-1]}. Failure mode: LOCAL MAXIMUM")
            return path, current, "local_max"


In [ ]:
# PART (a): test diagnose_hc on 3 custom landscapes
print("PART (a) - diagnose_hc: Detecting Three HC Failure Modes\n")

# Landscape 1: triggers LOCAL MAXIMUM
# state 2 (f=5) is both the global max and a local max
ls_lm = [3, 5, 2, 4, 1]

print("--- Failure Mode 1: LOCAL MAXIMUM ---")
print(f"  Landscape : {ls_lm}   (5 states)")
print(f"  State     :  1   2   3   4   5")
print(f"  f(s)      :  3   5   2   4   1")
print(f"  Global max: state 2 (f=5).")
print(f"  Start = 1 (f=3). Right s2(f=5) is better -> moves to s2.")
print(f"  At s2(f=5): left=s1(f=3) worse, right=s3(f=2) worse -> stuck.")
print(f"  f(current)=5 equals global max=5 -> LOCAL MAXIMUM termination.")
path, term, mode = diagnose_hc(ls_lm, 1)
print(f"  Path taken  : {path}")
print(f"  Mode        : {mode.upper()}")

PART (a) - diagnose_hc: Detecting Three HC Failure Modes

--- Failure Mode 1: LOCAL MAXIMUM ---
  Landscape : [3, 5, 2, 4, 1]   (5 states)
  State     :  1   2   3   4   5
  f(s)      :  3   5   2   4   1
  Global max: state 2 (f=5).
  Start = 1 (f=3). Right s2(f=5) is better -> moves to s2.
  At s2(f=5): left=s1(f=3) worse, right=s3(f=2) worse -> stuck.
  f(current)=5 equals global max=5 -> LOCAL MAXIMUM termination.
  Terminated at state 2 with f=5. Failure mode: LOCAL MAXIMUM
  Path taken  : [1, 2]
  Mode        : LOCAL_MAX


In [ ]:
# Landscape 2: triggers PLATEAU
# states 3, 4, 5 all have f=12 (flat plateau)
ls_pl = [4, 8, 12, 12, 12, 9]

print("--- Failure Mode 2: PLATEAU ---")
print(f"  Landscape : {ls_pl}   (6 states)")
print(f"  State     :  1   2   3   4   5   6")
print(f"  f(s)      :  4   8  12  12  12   9")
print(f"  States 3, 4, 5 form a flat plateau at f=12.")
print(f"  Start = 2 (f=8). Right s3(f=12) is better -> moves to s3.")
print(f"  At s3(f=12): left=s2(f=8) worse, right=s4(f=12) EQUAL -> PLATEAU.")
path, term, mode = diagnose_hc(ls_pl, 2)
print(f"  Path taken  : {path}")
print(f"  Mode        : {mode.upper()}")

--- Failure Mode 2: PLATEAU ---
  Landscape : [4, 8, 12, 12, 12, 9]   (6 states)
  State     :  1   2   3   4   5   6
  f(s)      :  4   8  12  12  12   9
  States 3, 4, 5 form a flat plateau at f=12.
  Start = 2 (f=8). Right s3(f=12) is better -> moves to s3.
  At s3(f=12): left=s2(f=8) worse, right=s4(f=12) EQUAL -> PLATEAU.
  Terminated at state 3 with f=12. Failure mode: PLATEAU
  Path taken  : [2, 3]
  Mode        : PLATEAU


In [ ]:
# Landscape 3: triggers RIDGE
# two peaks s3(f=8) and s5(f=9) separated by valley s4(f=6)
ls_rg = [2, 5, 8, 6, 9, 4, 2]

print("--- Failure Mode 3: RIDGE ---")
print(f"  Landscape : {ls_rg}   (7 states)")
print(f"  State     :  1   2   3   4   5   6   7")
print(f"  f(s)      :  2   5   8   6   9   4   2")
print(f"  Two peaks: s3(f=8) and s5(f=9) separated by valley s4(f=6).")
print(f"  Start = 4 (f=6). Left s3(f=8) is better -> moves to s3.")
print(f"  At s3(f=8): left=s2(f=5) worse, right=s4(f=6) worse -> stuck.")
print(f"  global_max=9 at s5. Valley at s4(f=6): 6 < 8 and 6 < 9 -> RIDGE.")
print(f"  Reaching s5(f=9) requires going DOWNHILL through s4(f=6) -- forbidden.")
path, term, mode = diagnose_hc(ls_rg, 4)
print(f"  Path taken  : {path}")
print(f"  Mode        : {mode.upper()}")


--- Failure Mode 3: RIDGE ---
  Landscape : [2, 5, 8, 6, 9, 4, 2]   (7 states)
  State     :  1   2   3   4   5   6   7
  f(s)      :  2   5   8   6   9   4   2
  Two peaks: s3(f=8) and s5(f=9) separated by valley s4(f=6).
  Start = 4 (f=6). Left s3(f=8) is better -> moves to s3.
  At s3(f=8): left=s2(f=5) worse, right=s4(f=6) worse -> stuck.
  global_max=9 at s5. Valley at s4(f=6): 6 < 8 and 6 < 9 -> RIDGE.
  Reaching s5(f=9) requires going DOWNHILL through s4(f=6) -- forbidden.
  Terminated at state 3 with f=8. Failure mode: RIDGE
  Path taken  : [4, 3]
  Mode        : RIDGE


In [ ]:
print("""
  Why HC cannot recover from a Local Maximum:
    At a local maximum every adjacent state has a strictly lower f-value,
    so the uphill condition (f(neighbour) > f(current)) is never satisfied.
    HC has no mechanism to move downhill, so it stays permanently at the peak.
    When the local maximum IS the global maximum the algorithm has succeeded;
    when it is sub-optimal (missed by directional bias), only random restarts
    or a downhill-tolerating algorithm (e.g. Simulated Annealing) can help.
""")



  Why HC cannot recover from a Local Maximum:
    At a local maximum every adjacent state has a strictly lower f-value,
    so the uphill condition (f(neighbour) > f(current)) is never satisfied.
    HC has no mechanism to move downhill, so it stays permanently at the peak.
    When the local maximum IS the global maximum the algorithm has succeeded;
    when it is sub-optimal (missed by directional bias), only random restarts
    or a downhill-tolerating algorithm (e.g. Simulated Annealing) can help.



In [ ]:
print("""
  Why HC cannot recover from a Plateau:
    On a plateau the strict uphill condition (f(neighbour) > f(current)) is
    never satisfied because all reachable neighbours have the same f-value.
    The algorithm stops at the first plateau state even though better regions
    may exist beyond the flat area. Sideways moves (with a cap to prevent
    infinite loops) are the only standard fix -- basic HC does not include them.
""")



  Why HC cannot recover from a Plateau:
    On a plateau the strict uphill condition (f(neighbour) > f(current)) is
    never satisfied because all reachable neighbours have the same f-value.
    The algorithm stops at the first plateau state even though better regions
    may exist beyond the flat area. Sideways moves (with a cap to prevent
    infinite loops) are the only standard fix -- basic HC does not include them.



In [ ]:
print("""
  Why HC cannot recover from a Ridge:
    A ridge occurs when the global maximum sits across a valley from the
    current local peak. The algorithm identifies that all neighbours of the
    current state are worse, but the only route to the higher peak requires
    moving downhill through the valley first -- which HC strictly forbids.
    The 1D neighbourhood is blind to peaks on the other side of a valley.
    Random restarts are the standard escape: a new random start may land on
    the correct side of the valley and climb directly to the global maximum.
""")



  Why HC cannot recover from a Ridge:
    A ridge occurs when the global maximum sits across a valley from the
    current local peak. The algorithm identifies that all neighbours of the
    current state are worse, but the only route to the higher peak requires
    moving downhill through the valley first -- which HC strictly forbids.
    The 1D neighbourhood is blind to peaks on the other side of a valley.
    Random restarts are the standard escape: a new random start may land on
    the correct side of the valley and climb directly to the global maximum.



In [ ]:
# PART (b): N-Queens Problem (N=8) with Stochastic HC + Random Restarts

# count_conflicts: counts attacking pairs of queens
def count_conflicts(board):
    """
    board[i] = row of queen in column i
    A conflict is when two queens are in the same row or on the same diagonal.
    Returns number of conflicting pairs (lower is better, 0 = solved).
    """
    conflicts = 0
    n = len(board)

    # compare every pair of queens
    for i in range(n):
        for j in range(i + 1, n):
            # same row?
            if board[i] == board[j]:
                conflicts = conflicts + 1
            # same diagonal?
            elif abs(board[i] - board[j]) == abs(i - j):
                conflicts = conflicts + 1

    return conflicts


In [ ]:
# Stochastic HC for N-Queens using column-based neighbourhood
# for each column, we try placing the queen in every possible row
def stochastic_hc_nqueens(board):
    """
    At each step:
    1. Try all column-row placements (N*N = 64 for N=8)
    2. Collect all moves that reduce conflict count
    3. If any improving moves exist, pick one randomly
    4. If no improving move, stop (stuck at local min)
    """
    current = list(board)  # make a copy
    current_conflicts = count_conflicts(current)
    n = len(current)

    while current_conflicts > 0:
        # find all improving moves
        improving = []

        for col in range(n):
            for row in range(n):
                # skip if queen is already in this row
                if row == current[col]:
                    continue

                # try placing queen in column col at row row
                new_board = list(current)
                new_board[col] = row
                new_conflicts = count_conflicts(new_board)

                # is this better?
                if new_conflicts < current_conflicts:
                    improving.append((list(new_board), new_conflicts))

        # if no improving move found, we are stuck
        if len(improving) == 0:
            break

        # pick one improving move randomly (stochastic)
        chosen = random.choice(improving)
        current = chosen[0]
        current_conflicts = chosen[1]

    return current, current_conflicts


In [ ]:
# generate a random starting board for N-Queens
def random_board(n=8):
    # one queen per column, placed in a random row
    board = []
    for col in range(n):
        board.append(random.randint(0, n - 1))
    return board

In [ ]:
# solve N-Queens using Random Restart HC
def solve_nqueens_rrhc(num_restarts, n=8):
    best_board = None
    best_conflicts = 99999  # start with a very large number

    for restart in range(1, num_restarts + 1):
        # generate a fresh random board
        start_board = random_board(n)
        board, conflicts = stochastic_hc_nqueens(start_board)

        # is this better than our best so far?
        if conflicts < best_conflicts:
            best_conflicts = conflicts
            best_board = list(board)

        # if we found a solution, stop early
        if conflicts == 0:
            return best_board, 0, restart

    return best_board, best_conflicts, num_restarts


In [ ]:
# print the board visually using Q and . characters
def print_board(board):
    n = len(board)
    print("  +" + "---+" * n)

    for row in range(n):
        line = "  |"
        for col in range(n):
            if board[col] == row:
                line = line + " Q |"
            else:
                line = line + " . |"
        print(line)
        print("  +" + "---+" * n)

    # print column numbers
    col_line = "  "
    for c in range(n):
        col_line = col_line + f"  {c+1} "
    print(col_line)


In [ ]:
# PART (b): Run N-Queens demo
print("PART (b) - N-Queens (N=8): Stochastic HC + Random Restarts\n")

print("""
Representation : board[i] = row of queen in column i  (list of 8 integers, rows 0-7).
Fitness        : count_conflicts(board) = attacking pairs (lower is better; 0 = solved).
Neighbourhood  : for each column, try all N possible row placements = N*N = 64 moves.
Move           : from all improving moves, pick one RANDOMLY (stochastic uphill).
Restart        : generate a fresh random board and re-run HC from scratch.
""")

# demonstrate count_conflicts on known boards
print("[b.1] Demonstrating count_conflicts on known boards:")
print("-" * 58)

demo_boards = [
    ([0, 1, 2, 3, 4, 5, 6, 7], "all on main diagonal -- 28 attacking pairs"),
    ([0, 4, 7, 5, 2, 6, 1, 3], "known 8-Queens solution"),
    ([3, 6, 2, 7, 1, 4, 0, 5], "another known 8-Queens solution"),
    ([0, 0, 0, 0, 0, 0, 0, 0], "all queens in row 0 -- 28 row conflicts"),
]

for i in range(len(demo_boards)):
    b = demo_boards[i][0]
    desc = demo_boards[i][1]
    c = count_conflicts(b)
    if c == 0:
        tag = "  <-- SOLUTION!"
    else:
        tag = ""
    print(f"  Board : {b}")
    print(f"  Conflicts = {c}{tag}   ({desc})")
    print()

# run the solver
print("[b.2] Running solve_nqueens_rrhc(100):")
random.seed(42)
solution, conflicts, restarts_used = solve_nqueens_rrhc(100, n=8)

print(f"  Restarts needed  : {restarts_used}")
print(f"  Final conflicts  : {conflicts}")
print(f"  Final board      : {solution}")

if conflicts == 0:
    print(f"  Result           : SOLUTION FOUND -- 0 conflicts!")
else:
    print(f"  Result           : Best found -- {conflicts} conflict(s) remain")

print()
print("  Visual board (row 0 = top, columns 1-8 = left to right):")
print_board(solution)


PART (b) - N-Queens (N=8): Stochastic HC + Random Restarts


Representation : board[i] = row of queen in column i  (list of 8 integers, rows 0-7).
Fitness        : count_conflicts(board) = attacking pairs (lower is better; 0 = solved).
Neighbourhood  : for each column, try all N possible row placements = N*N = 64 moves.
Move           : from all improving moves, pick one RANDOMLY (stochastic uphill).
Restart        : generate a fresh random board and re-run HC from scratch.

[b.1] Demonstrating count_conflicts on known boards:
----------------------------------------------------------
  Board : [0, 1, 2, 3, 4, 5, 6, 7]
  Conflicts = 28   (all on main diagonal -- 28 attacking pairs)

  Board : [0, 4, 7, 5, 2, 6, 1, 3]
  Conflicts = 0  <-- SOLUTION!   (known 8-Queens solution)

  Board : [3, 6, 2, 7, 1, 4, 0, 5]
  Conflicts = 0  <-- SOLUTION!   (another known 8-Queens solution)

  Board : [0, 0, 0, 0, 0, 0, 0, 0]
  Conflicts = 28   (all queens in row 0 -- 28 row conflicts)

[b.2] Running

In [ ]:
print("""
Explanation:
  Each column holds exactly one queen (guaranteed by the list representation).
  The column-based neighbourhood (N*N=64 moves) evaluates every possible row
  placement for every column at each step, dramatically reducing local minima
  compared to the pairwise-swap approach (only 28 possible moves).
  Stochastic selection (random.choice among all improving moves) avoids the
  deterministic bias of First-Choice, exploring more diverse paths through
  the conflict landscape.
  Random restarts escape local minima by generating a completely fresh random
  board whenever no improving move is available, allowing the search to try
  different regions of the conflict landscape.
""")



Explanation:
  Each column holds exactly one queen (guaranteed by the list representation).
  The column-based neighbourhood (N*N=64 moves) evaluates every possible row
  placement for every column at each step, dramatically reducing local minima
  compared to the pairwise-swap approach (only 28 possible moves).
  Stochastic selection (random.choice among all improving moves) avoids the
  deterministic bias of First-Choice, exploring more diverse paths through
  the conflict landscape.
  Random restarts escape local minima by generating a completely fresh random
  board whenever no improving move is available, allowing the search to try
  different regions of the conflict landscape.



In [ ]:
# PART (c): Benchmark the N-Queens solver
print("PART (c) - Benchmarking solve_nqueens_rrhc(k)  [N=8, 30 trials each]\n")

k_values = [5, 10, 25, 50, 100]
TRIALS = 30
n = 8

print(f"k values tested : {k_values}")
print(f"Trials per k    : {TRIALS}")
print(f"N (queens)      : {n}")
print(f"Neighbourhood   : column-based (N*N = 64 moves per step)")
print()

# go through each value of k
print(f"{'k':<8} {'Successes/30':<16} {'Success Rate':<16} {'Avg Restarts to Solution'}")
print("-" * 65)

for k in k_values:
    success_count = 0
    restart_counts_list = []  # only for successful runs

    for trial in range(TRIALS):
        random.seed(trial * 300 + k * 7)
        board, conflicts, used = solve_nqueens_rrhc(k, n=n)

        if conflicts == 0:
            success_count = success_count + 1
            restart_counts_list.append(used)

    # calculate success rate
    sr = success_count / TRIALS

    # calculate average restarts (only for successful runs)
    if len(restart_counts_list) > 0:
        total_restarts = 0
        for r in restart_counts_list:
            total_restarts = total_restarts + r
        avg_r = total_restarts / len(restart_counts_list)
        avg_str = f"{avg_r:.1f}"
    else:
        avg_str = "N/A"

    print(f"{k:<8} {str(success_count)+'/30':<16} {sr:<16.2%} {avg_str}")


PART (c) - Benchmarking solve_nqueens_rrhc(k)  [N=8, 30 trials each]

k values tested : [5, 10, 25, 50, 100]
Trials per k    : 30
N (queens)      : 8
Neighbourhood   : column-based (N*N = 64 moves per step)

k        Successes/30     Success Rate     Avg Restarts to Solution
-----------------------------------------------------------------
5        17/30            56.67%           2.5
10       25/30            83.33%           4.2
25       28/30            93.33%           7.3
50       30/30            100.00%          7.6
100      30/30            100.00%          6.7


In [ ]:
print("""
Analysis (5-6 sentences):

  Reason 1 -- Local minima in the conflict landscape:
  The N-Queens conflict landscape contains many local minima where every
  possible column-row placement either keeps the conflict count the same
  or raises it, yet the board still has 1-2 conflicts remaining. A structural
  property of the 8x8 conflict geometry that traps stochastic HC
  regardless of how many improving steps preceded the trap.

  Reason 2 -- Shrinking improving neighbourhood near solutions:
  As conflicts reduce from the initial count toward 1, the fraction of
  available improving moves (out of 64 column-row combinations) shrinks
  dramatically. Near a solution almost all moves are neutral or harmful.

  Experimental confirmation:
  The benchmark table shows success rate rising clearly with k: from 56.67%
  at k=5 to 100% at k=50 and k=100, directly confirming that more restarts
  overcome local minima by providing fresh random starting boards.

  Remaining limitation for N=1000:
  For N=1000 the search space is 1000^1000 possible boards and each HC step
  must evaluate N*N=1,000,000 column-row moves, making each restart extremely
  slow. The MIN-CONFLICTS algorithm is far more suitable: it directly targets
  the most conflicted queen and moves it to the row that minimises its conflicts,
  solving N=1,000,000 in near-linear time by exploiting domain-specific
  structure that RRHC completely lacks.
""")



Analysis (5-6 sentences):

  Reason 1 -- Deceptive local minima in the conflict landscape:
  The N-Queens conflict landscape contains many local minima where every
  possible column-row placement either keeps the conflict count the same
  or raises it, yet the board still has 1-2 conflicts remaining. These
  dead-end states arise because fixing the last conflicting pair by moving
  one queen inevitably introduces new conflicts between previously non-attacking
  queens -- a structural property of the 8x8 conflict geometry that traps
  stochastic HC regardless of how many improving steps preceded the trap.

  Reason 2 -- Shrinking improving neighbourhood near solutions:
  As conflicts reduce from the initial count toward 1, the fraction of
  available improving moves (out of 64 column-row combinations) shrinks
  dramatically. Near a solution almost all moves are neutral or harmful --
  the algorithm navigates an increasingly narrow funnel where the exact move
  needed to eliminate the l